In [4]:
from pymongo import MongoClient

# 1. Conectar al contenedor de Mongo
client = MongoClient('mongodb://mongodb:27017/')

# 2. Crear (o usar) la base de datos y la colección
db = client['Canasta_db'] 
coleccion = db['Retail_A'] # Ideal el nombre del grupo ejem: 'G_Ecommerce_scraping'

print("Conexión exitosa a MongoDB")

Conexión exitosa a MongoDB


In [2]:
# --- PASO 0: LIMPIEZA TOTAL Y REPARACIÓN ---
import os
import time
from pymongo import MongoClient
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# Cierra procesos viejos que hayan quedado abiertos
os.system("pkill -9 chrome")
os.system("pkill -9 chromedriver")
os.system("rm -rf /tmp/.com.google.Chrome.*")
os.system("rm -rf /tmp/.org.chromium.Chromium.*")
print("🧹 Limpieza de procesos y temporales completada.")

# --- VARIABLES GENERALES ---
NOMBRE_GRUPO = "Vannessa"
datos_finales = []   # Se define fuera del try para que siempre exista
driver = None        # Se define fuera del try para poder cerrarlo con seguridad

# --- PASO 1: CONFIGURACIÓN DEL NAVEGADOR ---
options = Options()
options.binary_location = "/usr/bin/google-chrome"  # Ruta del binario de Chrome

# Argumentos de estabilidad para Docker
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--disable-gpu")
options.add_argument("--disable-software-rasterizer")
options.add_argument("--window-size=1920,1080")
options.add_argument("--remote-debugging-port=9222")

# User-Agent común
options.add_argument(
    "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
    "AppleWebKit/537.36 (KHTML, like Gecko) "
    "Chrome/120.0.0.0 Safari/537.36"
)

try:
    # Inicia el navegador
    driver = webdriver.Chrome(options=options)
    print(" Navegador iniciado correctamente.")

    # --- PASO 2: NAVEGACIÓN Y EXTRACCIÓN ---
    limite_paginas = 3
    driver.get("https://www.amazon.es/s?k=laptops")

    for nivel_pagina in range(limite_paginas):
        print(f"--- Procesando Página {nivel_pagina + 1} ---")

        # Espera fija para que la página termine de cargar
        time.sleep(10)

        # Espera explícita a que existan resultados
        WebDriverWait(driver, 20).until(
            EC.presence_of_all_elements_located(
                (By.CSS_SELECTOR, "div[data-component-type='s-search-result']")
            )
        )

        bloques = driver.find_elements(
            By.CSS_SELECTOR, "div[data-component-type='s-search-result']"
        )

        for bloque in bloques:
            try:
                nombre = bloque.find_element(By.TAG_NAME, "h2").text
                precio = bloque.find_element(By.CSS_SELECTOR, ".a-price-whole").text

                datos_finales.append({
                    "identificador": nombre,
                    "valor": precio,
                    "fecha_captura": time.strftime("%Y-%m-%d %H:%M:%S"),
                    "grupo": NOMBRE_GRUPO
                })
            except:
                # Si el bloque no tiene nombre o precio, se salta
                continue

        # Intenta avanzar a la siguiente página
        try:
            btn_sig = driver.find_element(By.CLASS_NAME, "s-pagination-next")
            driver.execute_script("arguments[0].click();", btn_sig)
            time.sleep(5)
        except:
            print("No se encontró el botón siguiente o ya es la última página.")
            break

    print(f" Extracción terminada: {len(datos_finales)} productos.")

except Exception as e:
    print(f" Error en Selenium: {e}")

finally:
    # Cierra el navegador solo si logró abrirse
    if driver is not None:
        try:
            driver.quit()
        except:
            pass

# --- PASO 3: GUARDAR EN MONGODB ---
try:
    client = MongoClient("mongodb", 27017, serverSelectionTimeoutMS=5000)
    db = client["TiendaBigData"]
    coleccion = db["AmazonLaptops"]

    if datos_finales:
        for d in datos_finales:
            # Limpia el valor antes de convertirlo
            v_limpio = str(d["valor"]).replace(".", "").replace(",", "").strip()
            d["valor"] = float(v_limpio) if v_limpio.isdigit() else 0.0

        coleccion.insert_many(datos_finales)
        print(" Datos cargados en MongoDB correctamente.")
    else:
        print(" No hay datos para guardar.")

except Exception as e:
    print(f" Error en MongoDB: {e}")
    

🧹 Limpieza de procesos y temporales completada.
 Navegador iniciado correctamente.
--- Procesando Página 1 ---
--- Procesando Página 2 ---
--- Procesando Página 3 ---
 Extracción terminada: 89 productos.
 Datos cargados en MongoDB correctamente.


In [21]:
import requests
import time
from pymongo import MongoClient

URL = "https://cugat.cl/categoria-producto/despensa/"
headers = {"User-Agent": "Mozilla/5.0"}

datos_finales = []
NOMBRE_GRUPO = "Ave Mayo"

try:
    print("🌐 Scrapeando SOLO DESPENSA...")

    pagina = 1

    while True:

        r = requests.get(URL + f"page/{pagina}/", headers=headers, timeout=20)

        if r.status_code != 200:
            print("❌ Error:", r.status_code)
            break

        html = r.text

        # si no hay productos, corta
        if "woocommerce-loop-product" not in html:
            print("📭 Fin de productos")
            break

        from bs4 import BeautifulSoup
        soup = BeautifulSoup(html, "html.parser")

        productos = soup.select("li.product")

        print(f"📦 Página {pagina} -> {len(productos)} productos")

        for p in productos:
            try:
                nombre = p.select_one(".woocommerce-loop-product__title")
                precio = p.select_one("span.price")

                if not nombre or not precio:
                    continue

                nombre = nombre.text.strip()
                precio = precio.text.strip()

                datos_finales.append({
                    "identificador": nombre,
                    "valor": precio,
                    "grupo": NOMBRE_GRUPO,
                    "supermercado": "Cugat",
                    "fecha": time.strftime("%Y-%m-%d %H:%M:%S")
                })

                print(nombre, "->", precio)

            except:
                continue

        pagina += 1
        time.sleep(1)

    print("📊 TOTAL DESPENSA:", len(datos_finales))

except Exception as e:
    print("❌ Error:", repr(e))

# --- MONGO ---
if datos_finales:
    client = MongoClient("mongodb", 27017)
    db = client["Canasta_db"]
    col = db["Retail_A"]
    col.insert_many(datos_finales)
    print("💾 Guardado OK")
else:
    print("⚠️ Sin datos")

🌐 Scrapeando SOLO DESPENSA...
📦 Página 1 -> 0 productos
📦 Página 2 -> 0 productos
📦 Página 3 -> 0 productos
📦 Página 4 -> 0 productos
📦 Página 5 -> 0 productos
📦 Página 6 -> 0 productos
📦 Página 7 -> 0 productos
📦 Página 8 -> 0 productos
📦 Página 9 -> 0 productos
📦 Página 10 -> 0 productos
❌ Error: 404
📊 TOTAL DESPENSA: 0
⚠️ Sin datos


In [22]:
# --- PASO 0: LIMPIEZA TOTAL Y REPARACIÓN ---
import os
import time
from pymongo import MongoClient
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By

os.system("pkill -9 chrome")
os.system("pkill -9 chromedriver")
os.system("rm -rf /tmp/.com.google.Chrome.*")
os.system("rm -rf /tmp/.org.chromium.Chromium.*")
print("🧹 Limpieza de procesos y temporales completada.")

# --- VARIABLES GENERALES ---
NOMBRE_GRUPO = "Ave Mayo"
datos_finales = []
driver = None

# --- PASO 1: CONFIGURACIÓN DEL NAVEGADOR ---
options = Options()
options.binary_location = "/usr/bin/google-chrome"

options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--disable-gpu")
options.add_argument("--disable-software-rasterizer")
options.add_argument("--window-size=1920,1080")
options.add_argument("--remote-debugging-port=9222")

options.add_argument(
    "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
    "AppleWebKit/537.36 (KHTML, like Gecko) "
    "Chrome/120.0.0.0 Safari/537.36"
)

try:
    driver = webdriver.Chrome(options=options)
    print("🌐 Navegador iniciado correctamente.")

    # --- PASO 2: NAVEGACIÓN ---
    URL = "https://cugat.cl/categoria-producto/despensa/"
    driver.get(URL)

    print("🌐 URL actual:", driver.current_url)

    time.sleep(8)

    # --- SCROLL PARA CARGAR PRODUCTOS (CLAVE EN CUGAT) ---
    for i in range(6):
        driver.execute_script("window.scrollBy(0, 2000)")
        time.sleep(2)

    # --- EXTRACCIÓN ---
    bloques = driver.find_elements(By.CSS_SELECTOR, "li.product")

    print(f"📦 productos detectados: {len(bloques)}")

    for bloque in bloques:
        try:
            # --- NOMBRE ---
            try:
                nombre = bloque.find_element(
                    By.CSS_SELECTOR,
                    ".woocommerce-loop-product__title"
                ).text.strip()
            except:
                nombre = bloque.text.strip()

            if not nombre:
                continue

            # --- PRECIO ---
            try:
                precio = bloque.find_element(
                    By.CSS_SELECTOR,
                    "span.price"
                ).text.strip()
            except:
                continue

            if not precio:
                continue

            datos_finales.append({
                "identificador": nombre,
                "valor": precio,
                "fecha_captura": time.strftime("%Y-%m-%d %H:%M:%S"),
                "grupo": NOMBRE_GRUPO
            })

            print(nombre, "->", precio)

        except:
            continue

    print(f"📊 Extracción terminada: {len(datos_finales)} productos.")

except Exception as e:
    print("❌ Error en Selenium:")
    print(repr(e))

finally:
    if driver is not None:
        try:
            driver.quit()
        except:
            pass

# --- PASO 3: GUARDAR EN MONGODB ---
try:
    client = MongoClient("mongodb", 27017, serverSelectionTimeoutMS=5000)
    db = client["Canasta_db"]
    coleccion = db["Retail_A"]

    if datos_finales:

        # limpiar precios
        for d in datos_finales:
            v_limpio = str(d["valor"]).replace("$", "").replace(".", "").replace(",", "").strip()
            d["valor"] = int(v_limpio) if v_limpio.isdigit() else 0

        coleccion.insert_many(datos_finales)
        print("💾 Datos cargados en MongoDB correctamente.")

    else:
        print("⚠️ No hay datos para guardar.")

except Exception as e:
    print("❌ Error en MongoDB:")
    print(repr(e))

🧹 Limpieza de procesos y temporales completada.
🌐 Navegador iniciado correctamente.
🌐 URL actual: https://cugat.cl/categoria-producto/despensa/
📦 productos detectados: 0
📊 Extracción terminada: 0 productos.
⚠️ No hay datos para guardar.


In [24]:
# --- PASO 0: LIMPIEZA TOTAL ---
import os
import time
from pymongo import MongoClient
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By

os.system("pkill -9 chrome")
os.system("pkill -9 chromedriver")
os.system("rm -rf /tmp/.com.google.Chrome.*")
os.system("rm -rf /tmp/.org.chromium.Chromium.*")
print("🧹 Limpieza completada")

# --- VARIABLES ---
NOMBRE_SUPER = "Mayorista 10"
NOMBRE_GRUPO = "Ave Mayo"
datos = []

# productos base canasta básica (IMPORTANTE)
productos_objetivo = [
    "arroz",
    "harina",
    "azúcar",
    "aceite",
    "leche",
    "fideos",
    "huevos"
]

# --- PASO 1: CONFIG SELENIUM ---
options = Options()
options.binary_location = "/usr/bin/google-chrome"

options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--disable-gpu")
options.add_argument("--window-size=1920,1080")

driver = webdriver.Chrome(options=options)

try:
    driver.get("https://www.mayorista10.cl/")
    print("🌐 Sitio abierto")

    time.sleep(10)

    # scroll para cargar productos
    for i in range(5):
        driver.execute_script("window.scrollBy(0, 1500)")
        time.sleep(2)

    # --- EXTRACCIÓN ---
    cards = driver.find_elements(By.XPATH, "//a[contains(@href,'/producto')]")

    print(f"📦 productos detectados: {len(cards)}")

    for c in cards:
        try:
            texto = c.text.strip()
            if texto == "":
                continue

            # nombre
            nombre = texto.split("\n")[0]

            # filtro canasta básica (CLAVE PARA TU PROYECTO)
            if not any(p in nombre.lower() for p in productos_objetivo):
                continue

            # precio
            try:
                precio = c.find_element(By.XPATH, ".//*[contains(text(),'$')]").text
            except:
                continue

            precio_limpio = int(
                precio.replace("$", "")
                      .replace(".", "")
                      .replace(",", "")
            )

            datos.append({
                "producto": nombre,
                "precio": precio_limpio,
                "supermercado": NOMBRE_SUPER,
                "grupo": NOMBRE_GRUPO,
                "fecha": time.strftime("%Y-%m-%d %H:%M:%S")
            })

            print(nombre, "->", precio_limpio)

        except:
            continue

    print(f"📊 Total canasta básica: {len(datos)}")

except Exception as e:
    print("❌ Error:", repr(e))

finally:
    driver.quit()

# --- MONGO ---
try:
    client = MongoClient("mongodb", 27017)
    db = client["Canasta_db"]
    col = db["Retail_A"]

    if datos:
        col.insert_many(datos)
        print("💾 Guardado en Mongo OK")
    else:
        print("⚠️ Sin datos para guardar")

except Exception as e:
    print("❌ Mongo error:", repr(e))

🧹 Limpieza completada
🌐 Sitio abierto
📦 productos detectados: 0
📊 Total canasta básica: 0
⚠️ Sin datos para guardar


In [25]:
# --- PASO 0 ---
import os
import time
from pymongo import MongoClient
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By

os.system("pkill -9 chrome")
os.system("pkill -9 chromedriver")
print("🧹 Limpieza OK")

# --- CONFIG ---
datos = []

options = Options()
options.binary_location = "/usr/bin/google-chrome"
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--window-size=1920,1080")

driver = webdriver.Chrome(options=options)

try:
    driver.get("https://www.mayorista10.cl/")
    print("🌐 Sitio cargado")

    time.sleep(10)

    # scroll para cargar productos
    for i in range(6):
        driver.execute_script("window.scrollBy(0, 1500)")
        time.sleep(2)

    # 🔥 CLAVE: cards reales (no links)
    cards = driver.find_elements(By.CSS_SELECTOR, "a")

    print("🔎 total elementos:", len(cards))

    for c in cards:
        try:
            texto = c.text.strip()

            # filtra basura
            if len(texto) < 10:
                continue

            # intenta encontrar precio dentro del mismo bloque
            try:
                precio = c.find_element(By.XPATH, ".//*[contains(text(),'$')]").text
            except:
                continue

            nombre = texto.split("\n")[0]

            precio_limpio = int(
                precio.replace("$", "")
                      .replace(".", "")
                      .replace(",", "")
            )

            datos.append({
                "producto": nombre,
                "precio": precio_limpio
            })

            print(nombre, "->", precio_limpio)

        except:
            continue

    print("📊 TOTAL:", len(datos))

except Exception as e:
    print("❌ Error:", repr(e))

finally:
    driver.quit()

🧹 Limpieza OK
🌐 Sitio cargado
🔎 total elementos: 59
📊 TOTAL: 0


In [26]:
# --- PASO 0: LIMPIEZA TOTAL ---
import os
import time
from pymongo import MongoClient
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By

os.system("pkill -9 chrome")
os.system("pkill -9 chromedriver")
os.system("rm -rf /tmp/.com.google.Chrome.*")
os.system("rm -rf /tmp/.org.chromium.Chromium.*")
print("🧹 Limpieza completada")

# --- VARIABLES ---
datos = []
NOMBRE_SUPER = "Mayorista 10"
NOMBRE_GRUPO = "Canasta Básica"

productos_objetivo = [
    "arroz",
    "harina",
    "azúcar",
    "aceite",
    "leche",
    "fideos",
    "huevos"
]

# --- PASO 1: SELENIUM ---
options = Options()
options.binary_location = "/usr/bin/google-chrome"

options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--window-size=1920,1080")

driver = webdriver.Chrome(options=options)

try:
    driver.get("https://www.mayorista10.cl/")
    print("🌐 Sitio cargado")

    time.sleep(10)

    # scroll para cargar contenido
    for i in range(6):
        driver.execute_script("window.scrollBy(0, 1500)")
        time.sleep(2)

    # 🔥 DEBUG IMPORTANTE
    print("🔍 HTML parcial:", driver.page_source[:500])

    # --- EXTRACCIÓN ---
    elementos = driver.find_elements(By.XPATH, "//*[contains(text(),'$')]")

    print(f"📦 elementos con precio detectados: {len(elementos)}")

    for e in elementos:
        try:
            texto = e.text.strip()
            if texto == "":
                continue

            # buscar nombre cercano (heurística simple)
            try:
                contenedor = e.find_element(By.XPATH, "./ancestor::*[1]")
                nombre = contenedor.text.split("\n")[0]
            except:
                nombre = "sin nombre"

            # filtro canasta básica
            if not any(p in nombre.lower() for p in productos_objetivo):
                continue

            precio_limpio = int(
                texto.replace("$", "")
                     .replace(".", "")
                     .replace(",", "")
                     .strip()
            )

            datos.append({
                "producto": nombre,
                "precio": precio_limpio,
                "supermercado": NOMBRE_SUPER,
                "grupo": NOMBRE_GRUPO,
                "fecha": time.strftime("%Y-%m-%d %H:%M:%S")
            })

            print(nombre, "->", precio_limpio)

        except:
            continue

    print("📊 TOTAL PRODUCTOS:", len(datos))

except Exception as e:
    print("❌ Error Selenium:")
    print(repr(e))

finally:
    driver.quit()

# --- PASO 3: MONGO ---
try:
    client = MongoClient("mongodb", 27017, serverSelectionTimeoutMS=5000)

    db = client["Canasta_db"]
    coleccion = db["Retail_A"]

    print("🧠 Conectando a MongoDB...")

    if len(datos) > 0:

        for d in datos:
            try:
                d["precio"] = int(str(d["precio"]))
            except:
                d["precio"] = 0

        resultado = coleccion.insert_many(datos)

        print("💾 Guardados en MongoDB:", len(resultado.inserted_ids))
        print("📦 DB: Canasta_db")
        print("📁 Colección: Retail_A")

    else:
        print("⚠️ No hay datos para guardar (scraping falló)")

except Exception as e:
    print("❌ Error MongoDB:")
    print(repr(e))

🧹 Limpieza completada
🌐 Sitio cargado
🔍 HTML parcial: <html><head><meta charset="utf-8" data-next-head=""><meta name="viewport" content="width=device-width" data-next-head=""><title data-next-head="">Mayorista10 Precios al chancho para todos</title><meta name="description" content="¡Compra en Supermercados Mayorista10 y vive la nueva experiencia del ahorro! Revisa nuestro catálogo de ofertas y encuentra tu local más cercano. " data-next-head=""><meta property="og:title" content="Mayorista10 Precios al chancho para todos" data-next-head=""><meta pro
📦 elementos con precio detectados: 12
Fideos Merkat Variedades 400 G -> 450
Leche En Polvo Colun Entera Instantanea 900 Gr -> 6550
📊 TOTAL PRODUCTOS: 2
🧠 Conectando a MongoDB...
💾 Guardados en MongoDB: 2
📦 DB: Canasta_db
📁 Colección: Retail_A


In [27]:
# --- PASO 0: LIMPIEZA TOTAL ---
import os
import time
from pymongo import MongoClient
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By

os.system("pkill -9 chrome")
os.system("pkill -9 chromedriver")
os.system("rm -rf /tmp/.com.google.Chrome.*")
os.system("rm -rf /tmp/.org.chromium.Chromium.*")
print("🧹 Limpieza completada")

# --- VARIABLES ---
datos = []
NOMBRE_SUPER = "Mayorista 10"

# 🔥 FILTRO ALIMENTICIO AMPLIO
palabras_alimentacion = [
    "arroz", "leche", "azúcar", "azucar", "harina", "aceite",
    "fideos", "pasta", "cereal", "galleta", "chocolate",
    "carne", "pollo", "cerdo", "vacuno", "huevo",
    "pan", "queso", "yogurt", "yoghurt", "mantequilla",
    "bebida", "jugo", "te", "té", "café", "salsa",
    "conserva", "atun", "atún", "snack", "snacks"
]

# --- SELENIUM ---
options = Options()
options.binary_location = "/usr/bin/google-chrome"

options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--window-size=1920,1080")

driver = webdriver.Chrome(options=options)

try:
    driver.get("https://www.mayorista10.cl/")
    print("🌐 Sitio cargado")

    time.sleep(10)

    # scroll para cargar productos
    for i in range(6):
        driver.execute_script("window.scrollBy(0, 1500)")
        time.sleep(2)

    # 🔍 debug opcional
    print("🔍 HTML parcial:", driver.page_source[:400])

    # --- EXTRACCIÓN ---
    cards = driver.find_elements(By.XPATH, "//a[contains(@href,'producto')]")

    print("📦 productos detectados:", len(cards))

    for c in cards:
        try:
            texto = c.text.strip()

            if texto == "":
                continue

            nombre = texto.split("\n")[0]
            nombre_lower = nombre.lower()

            # 🔥 filtro alimenticio
            if not any(p in nombre_lower for p in palabras_alimentacion):
                continue

            # precio
            try:
                precio = c.find_element(By.XPATH, ".//*[contains(text(),'$')]").text
            except:
                continue

            precio_limpio = int(
                precio.replace("$", "")
                      .replace(".", "")
                      .replace(",", "")
                      .strip()
            )

            datos.append({
                "producto": nombre,
                "precio": precio_limpio,
                "supermercado": NOMBRE_SUPER,
                "tipo": "alimenticio",
                "fecha": time.strftime("%Y-%m-%d %H:%M:%S")
            })

            print(nombre, "->", precio_limpio)

        except:
            continue

    print("📊 TOTAL PRODUCTOS:", len(datos))

except Exception as e:
    print("❌ Error Selenium:")
    print(repr(e))

finally:
    driver.quit()

# --- MONGO ---
try:
    client = MongoClient("mongodb", 27017, serverSelectionTimeoutMS=5000)

    db = client["Canasta_db"]
    coleccion = db["Retail_A"]

    print("🧠 Conectando a MongoDB...")

    if len(datos) > 0:

        # limpieza segura
        for d in datos:
            try:
                d["precio"] = int(str(d["precio"]))
            except:
                d["precio"] = 0

        resultado = coleccion.insert_many(datos)

        print("💾 Guardados en MongoDB:", len(resultado.inserted_ids))
        print("📦 DB: Canasta_db")
        print("📁 Colección: Retail_A")

    else:
        print("⚠️ No hay datos para guardar (scraping vacío)")

except Exception as e:
    print("❌ Error MongoDB:")
    print(repr(e))

🧹 Limpieza completada
🌐 Sitio cargado
🔍 HTML parcial: <html><head><meta charset="utf-8" data-next-head=""><meta name="viewport" content="width=device-width" data-next-head=""><title data-next-head="">Mayorista10 Precios al chancho para todos</title><meta name="description" content="¡Compra en Supermercados Mayorista10 y vive la nueva experiencia del ahorro! Revisa nuestro catálogo de ofertas y encuentra tu local más cercano. " data-next-head=""><meta
📦 productos detectados: 0
📊 TOTAL PRODUCTOS: 0
🧠 Conectando a MongoDB...
⚠️ No hay datos para guardar (scraping vacío)


In [28]:
# --- PASO 0: LIMPIEZA ---
import os
import time
from pymongo import MongoClient
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By

os.system("pkill -9 chrome")
os.system("pkill -9 chromedriver")
print("🧹 Limpieza completada")

# --- VARIABLES ---
datos = []
NOMBRE_SUPER = "Mayorista 10"

# filtro alimenticio amplio
palabras_alimentacion = [
    "arroz", "leche", "azúcar", "harina", "aceite",
    "fideos", "pasta", "cereal", "galleta", "chocolate",
    "carne", "pollo", "huevo", "pan", "queso",
    "yogurt", "mantequilla", "bebida", "jugo", "café",
    "té", "salsa", "atun", "atún", "snack"
]

# --- SELENIUM ---
options = Options()
options.binary_location = "/usr/bin/google-chrome"
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--window-size=1920,1080")

driver = webdriver.Chrome(options=options)

try:
    url = "https://www.mayorista10.cl/"
    driver.get(url)

    print("🌐 Sitio cargado")
    time.sleep(10)

    # scroll (por si hay lazy load)
    for i in range(6):
        driver.execute_script("window.scrollBy(0, 1500)")
        time.sleep(2)

    # 🔍 DEBUG REAL
    html = driver.page_source.lower()

    if "producto" not in html:
        print("⚠️ ALERTA: No se detectan productos en el HTML")
        print("👉 Esto significa que la web carga datos por API (React/Next.js)")
        print("👉 Selenium NO puede ver los productos aquí")
    else:
        print("✔ Se detecta estructura de productos")

    # intento extracción
    cards = driver.find_elements(By.XPATH, "//a[contains(@href,'producto')]")

    print("📦 elementos detectados:", len(cards))

    for c in cards:
        try:
            texto = c.text.strip()
            if texto == "":
                continue

            nombre = texto.split("\n")[0]
            nombre_lower = nombre.lower()

            # filtro alimenticio
            if not any(p in nombre_lower for p in palabras_alimentacion):
                continue

            try:
                precio = c.find_element(By.XPATH, ".//*[contains(text(),'$')]").text
            except:
                continue

            precio_limpio = int(
                precio.replace("$", "")
                      .replace(".", "")
                      .replace(",", "")
                      .strip()
            )

            datos.append({
                "producto": nombre,
                "precio": precio_limpio,
                "supermercado": NOMBRE_SUPER,
                "fecha": time.strftime("%Y-%m-%d %H:%M:%S")
            })

            print(nombre, "->", precio_limpio)

        except:
            continue

    print("📊 TOTAL PRODUCTOS:", len(datos))

except Exception as e:
    print("❌ Error Selenium:", repr(e))

finally:
    driver.quit()

# --- MONGO ---
try:
    client = MongoClient("mongodb", 27017, serverSelectionTimeoutMS=5000)
    db = client["Canasta_db"]
    col = db["Retail_A"]

    print("🧠 Conectando a MongoDB...")

    if len(datos) > 0:

        for d in datos:
            try:
                d["precio"] = int(str(d["precio"]))
            except:
                d["precio"] = 0

        col.insert_many(datos)

        print("💾 Guardados en MongoDB:", len(datos))
        print("📦 DB: Canasta_db")
        print("📁 Colección: Retail_A")

    else:
        print("⚠️ No hay datos para guardar")
        print("👉 causa real: Mayorista 10 carga productos por API (no HTML)")

except Exception as e:
    print("❌ Error MongoDB:", repr(e))

🧹 Limpieza completada
🌐 Sitio cargado
✔ Se detecta estructura de productos
📦 elementos detectados: 0
📊 TOTAL PRODUCTOS: 0
🧠 Conectando a MongoDB...
⚠️ No hay datos para guardar
👉 causa real: Mayorista 10 carga productos por API (no HTML)


In [29]:
# --- LIMPIEZA ---
import os
import time
import re
from pymongo import MongoClient
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

os.system("pkill -9 chrome")
os.system("pkill -9 chromedriver")
print("🧹 Limpieza completada")

# --- CONFIG ---
URL = "https://cugat.cl/categoria-producto/despensa/"
datos = []

# filtro alimentos
palabras_alimentacion = [
    "arroz","leche","harina","aceite","azúcar","azucar",
    "fideos","pasta","galleta","chocolate","cereal",
    "carne","pollo","huevo","queso","yogurt","mantequilla",
    "bebida","jugo","café","te","té","atun","atún","salsa"
]

# --- SELENIUM ---
options = Options()
options.binary_location = "/usr/bin/google-chrome"
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--window-size=1920,1080")

driver = webdriver.Chrome(options=options)

try:
    driver.get(URL)
    print("🌐 Cugat cargado")

    # espera carga JS
    time.sleep(8)

    # scroll para cargar productos
    for i in range(5):
        driver.execute_script("window.scrollBy(0, 1500)")
        time.sleep(2)

    # esperar productos WooCommerce
    WebDriverWait(driver, 15).until(
        EC.presence_of_all_elements_located(
            (By.CSS_SELECTOR, "a.woocommerce-LoopProduct-link")
        )
    )

    productos = driver.find_elements(By.CSS_SELECTOR, "a.woocommerce-LoopProduct-link")

    print(f"📦 productos detectados: {len(productos)}")

    for p in productos:
        try:
            nombre = p.text.strip().lower()

            if nombre == "":
                continue

            # filtro alimentos
            if not any(x in nombre for x in palabras_alimentacion):
                continue

            # precio (ins = precio actual)
            try:
                precio = p.find_element(
                    By.XPATH,
                    ".//ins//span[contains(@class,'woocommerce-Price-amount')]"
                ).text
            except:
                continue

            # limpiar precio
            precio_limpio = int(
                re.sub(r'[^\d]', '', precio)
            )

            datos.append({
                "producto": nombre,
                "precio": precio_limpio,
                "supermercado": "Cugat",
                "fecha": time.strftime("%Y-%m-%d %H:%M:%S")
            })

            print(nombre, "->", precio_limpio)

        except:
            continue

    print("📊 TOTAL PRODUCTOS:", len(datos))

except Exception as e:
    print("❌ Error:", repr(e))

finally:
    driver.quit()

# --- MONGO ---
try:
    client = MongoClient("mongodb", 27017)
    db = client["Canasta_db"]
    col = db["Retail_A"]

    if len(datos) > 0:
        col.insert_many(datos)
        print("💾 Guardado en MongoDB OK")
    else:
        print("⚠️ No se encontraron productos alimenticios")

except Exception as e:
    print("❌ Mongo error:", repr(e))

🧹 Limpieza completada
🌐 Cugat cargado
📦 productos detectados: 30
📊 TOTAL PRODUCTOS: 0
⚠️ No se encontraron productos alimenticios


In [30]:
# --- LIMPIEZA ---
import os
import time
import re
from pymongo import MongoClient
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

os.system("pkill -9 chrome")
os.system("pkill -9 chromedriver")
print("🧹 Limpieza completada")

# --- CONFIG ---
URL = "https://cugat.cl/categoria-producto/carniceria/"
datos = []

# --- SELENIUM ---
options = Options()
options.binary_location = "/usr/bin/google-chrome"
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--window-size=1920,1080")

driver = webdriver.Chrome(options=options)

try:
    driver.get(URL)
    print("🌐 Cugat Carnicería cargado")

    time.sleep(8)

    # scroll para cargar todos los productos
    for i in range(6):
        driver.execute_script("window.scrollBy(0, 1500)")
        time.sleep(2)

    # esperar productos WooCommerce
    WebDriverWait(driver, 15).until(
        EC.presence_of_all_elements_located(
            (By.CSS_SELECTOR, "a.woocommerce-LoopProduct-link")
        )
    )

    productos = driver.find_elements(By.CSS_SELECTOR, "a.woocommerce-LoopProduct-link")

    print(f"📦 productos detectados: {len(productos)}")

    for p in productos:
        try:
            nombre = p.text.strip()

            if nombre == "":
                continue

            # precio actual (ins)
            try:
                precio = p.find_element(
                    By.XPATH,
                    ".//ins//span[contains(@class,'woocommerce-Price-amount')]"
                ).text
            except:
                continue

            precio_limpio = int(
                re.sub(r'[^\d]', '', precio)
            )

            datos.append({
                "producto": nombre,
                "precio": precio_limpio,
                "categoria": "carniceria",
                "supermercado": "Cugat",
                "fecha": time.strftime("%Y-%m-%d %H:%M:%S")
            })

            print(nombre, "->", precio_limpio)

        except:
            continue

    print("📊 TOTAL PRODUCTOS:", len(datos))

except Exception as e:
    print("❌ Error:", repr(e))

finally:
    driver.quit()

# --- MONGO ---
try:
    client = MongoClient("mongodb", 27017)
    db = client["Canasta_db"]
    col = db["Retail_A"]

    if len(datos) > 0:
        col.insert_many(datos)
        print("💾 Guardado en MongoDB OK")
    else:
        print("⚠️ No se encontraron productos")

except Exception as e:
    print("❌ Mongo error:", repr(e))

🧹 Limpieza completada
🌐 Cugat Carnicería cargado
📦 productos detectados: 30
📊 TOTAL PRODUCTOS: 0
⚠️ No se encontraron productos


In [31]:
# --- LIMPIEZA ---
import os
import time
import re
from pymongo import MongoClient
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

os.system("pkill -9 chrome")
os.system("pkill -9 chromedriver")
print("🧹 Limpieza completada")

# --- CONFIG ---
URL = "https://cugat.cl/categoria-producto/carniceria/"
datos = []

# --- SELENIUM ---
options = Options()
options.binary_location = "/usr/bin/google-chrome"
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--window-size=1920,1080")

driver = webdriver.Chrome(options=options)

try:
    driver.get(URL)
    print("🌐 Cugat Carnicería cargado")

    time.sleep(8)

    # scroll para cargar productos
    for i in range(6):
        driver.execute_script("window.scrollBy(0, 1500)")
        time.sleep(2)

    # esperar productos reales (contenedor correcto)
    WebDriverWait(driver, 15).until(
        EC.presence_of_all_elements_located(
            (By.CSS_SELECTOR, "div.title-wrapper")
        )
    )

    cards = driver.find_elements(By.CSS_SELECTOR, "div.title-wrapper")

    print(f"📦 productos detectados: {len(cards)}")

    for c in cards:
        try:
            # --- NOMBRE ---
            nombre = c.find_element(By.CSS_SELECTOR, "p.name a").text.strip()

            # --- PRECIO ---
            precio = c.find_element(By.CSS_SELECTOR, "span.price").text

            # limpiar precio principal ($8.690)
            match = re.search(r"\$[\d\.]+", precio)
            if not match:
                continue

            precio_limpio = int(
                match.group(0)
                .replace("$", "")
                .replace(".", "")
            )

            datos.append({
                "producto": nombre,
                "precio": precio_limpio,
                "categoria": "carniceria",
                "supermercado": "Cugat",
                "fecha": time.strftime("%Y-%m-%d %H:%M:%S")
            })

            print(nombre, "->", precio_limpio)

        except:
            continue

    print("📊 TOTAL PRODUCTOS:", len(datos))

except Exception as e:
    print("❌ Error:", repr(e))

finally:
    driver.quit()

# --- MONGO ---
try:
    client = MongoClient("mongodb", 27017)
    db = client["Canasta_db"]
    col = db["Retail_A"]

    if len(datos) > 0:
        col.insert_many(datos)
        print("💾 Guardado en MongoDB OK")
    else:
        print("⚠️ No se encontraron productos")

except Exception as e:
    print("❌ Mongo error:", repr(e))

🧹 Limpieza completada
🌐 Cugat Carnicería cargado
📦 productos detectados: 30
📊 TOTAL PRODUCTOS: 0
⚠️ No se encontraron productos


In [34]:
# --- LIMPIEZA SEGURA ---
import os
import time
import re
from pymongo import MongoClient
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait

os.system("pkill -9 chrome")
os.system("pkill -9 chromedriver")
print("🧹 Limpieza completada")

URL = "https://cugat.cl/categoria-producto/carniceria/"
datos = []

# --- CONFIG CHROME ---
def crear_driver():
    options = Options()
    options.binary_location = "/usr/bin/google-chrome"
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--window-size=1920,1080")
    options.add_argument("--disable-gpu")
    options.add_argument("--disable-extensions")

    return webdriver.Chrome(options=options)

# --- INICIO SEGURO DEL DRIVER ---
driver = None

try:
    driver = crear_driver()
    print("🌐 Driver iniciado")

    driver.get(URL)
    print("🌐 Página cargada")

    time.sleep(12)  # render real (clave en Cugat)

    # scroll progresivo para cargar productos
    for i in range(6):
        driver.execute_script("window.scrollBy(0, 1500)")
        time.sleep(2)

    # espera robusta (evita DOM vacío)
    WebDriverWait(driver, 25).until(
        lambda d: len(d.find_elements(By.CSS_SELECTOR, "div.title-wrapper")) > 0
    )

    cards = driver.find_elements(By.CSS_SELECTOR, "div.title-wrapper")

    print(f"📦 cards detectadas: {len(cards)}")

    # --- SCRAPING CON REINTENTO INTERNO ---
    for i in range(len(cards)):

        for intento in range(3):  # 🔥 retry por card
            try:
                card = cards[i]

                nombre = card.find_element(By.CSS_SELECTOR, "p.name a").text.strip()

                if nombre == "":
                    raise Exception("nombre vacío")

                precio_texto = card.find_element(By.CSS_SELECTOR, "span.price").text

                match = re.search(r"\$[\d\.]+", precio_texto)
                if not match:
                    raise Exception("sin precio")

                precio = int(match.group(0).replace("$", "").replace(".", ""))

                datos.append({
                    "producto": nombre,
                    "precio": precio,
                    "supermercado": "Cugat",
                    "categoria": "carniceria",
                    "fecha": time.strftime("%Y-%m-%d %H:%M:%S")
                })

                print(nombre, "->", precio)
                break  # éxito → salir retry

            except:
                time.sleep(2)
                continue

    print("📊 TOTAL PRODUCTOS:", len(datos))

except Exception as e:
    print("❌ ERROR GENERAL:", repr(e))

finally:
    # --- CIERRE SEGURO ---
    try:
        if driver:
            driver.quit()
            print("🔒 Driver cerrado correctamente")
    except:
        pass

# --- MONGO SEGURO ---
try:
    client = MongoClient("mongodb", 27017, serverSelectionTimeoutMS=5000)
    db = client["Canasta_db"]
    col = db["Retail_A"]

    print("🧠 Conectando a MongoDB...")

    if len(datos) > 0:
        col.insert_many(datos)
        print("💾 Guardados en MongoDB:", len(datos))
    else:
        print("⚠️ Sin datos para guardar")

except Exception as e:
    print("❌ Mongo error:", repr(e))

🧹 Limpieza completada
🌐 Driver iniciado
🌐 Página cargada
📦 cards detectadas: 30
📊 TOTAL PRODUCTOS: 0
🔒 Driver cerrado correctamente
🧠 Conectando a MongoDB...
⚠️ Sin datos para guardar


In [35]:
# --- LIMPIEZA ---
import os
import time
import re
from pymongo import MongoClient
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait

os.system("pkill -9 chrome")
os.system("pkill -9 chromedriver")
print("🧹 Limpieza completada")

URL = "https://cugat.cl/categoria-producto/carniceria/"
datos = []

def crear_driver():
    options = Options()
    options.binary_location = "/usr/bin/google-chrome"
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--window-size=1920,1080")
    return webdriver.Chrome(options=options)

driver = crear_driver()

try:
    driver.get(URL)
    print("🌐 Driver iniciado")

    time.sleep(15)  # 🔥 clave (Cugat render lento)

    for i in range(6):
        driver.execute_script("window.scrollBy(0, 1500)")
        time.sleep(2)

    cards = WebDriverWait(driver, 20).until(
        lambda d: d.find_elements(By.CSS_SELECTOR, "div.title-wrapper")
    )

    print(f"📦 cards detectadas: {len(cards)}")

    for card in cards:
        try:
            # 🔥 CAMBIO CLAVE: usar innerText en vez de .text
            raw_text = card.get_attribute("innerText")

            if not raw_text or raw_text.strip() == "":
                continue

            # separar líneas
            lineas = raw_text.split("\n")

            # buscar nombre (primera línea útil)
            nombre = None
            for l in lineas:
                if len(l) > 10:
                    nombre = l
                    break

            if not nombre:
                continue

            # buscar precio
            match = re.search(r"\$[\d\.]+", raw_text)
            if not match:
                continue

            precio = int(match.group(0).replace("$", "").replace(".", ""))

            datos.append({
                "producto": nombre.strip(),
                "precio": precio,
                "supermercado": "Cugat",
                "categoria": "carniceria",
                "fecha": time.strftime("%Y-%m-%d %H:%M:%S")
            })

            print(nombre, "->", precio)

        except:
            continue

    print("📊 TOTAL PRODUCTOS:", len(datos))

except Exception as e:
    print("❌ ERROR:", repr(e))

finally:
    driver.quit()
    print("🔒 Driver cerrado")

# --- MONGO ---
try:
    client = MongoClient("mongodb", 27017, serverSelectionTimeoutMS=5000)
    db = client["Canasta_db"]
    col = db["Retail_A"]

    if len(datos) > 0:
        col.insert_many(datos)
        print("💾 Guardados en MongoDB:", len(datos))
    else:
        print("⚠️ Sin datos para guardar")

except Exception as e:
    print("❌ Mongo error:", repr(e))

🧹 Limpieza completada
🌐 Driver iniciado
📦 cards detectadas: 30
📊 TOTAL PRODUCTOS: 0
🔒 Driver cerrado
⚠️ Sin datos para guardar


In [36]:
import requests
import time
import re
from pymongo import MongoClient

print("🌐 Scraping API COMPLETO Cugat")

datos = []
pagina = 1

while True:
    url = f"https://cugat.cl/wp-json/wc/store/products?per_page=100&page={pagina}"

    r = requests.get(url, timeout=15)

    if r.status_code != 200:
        print(f"❌ Fin o error en página {pagina}")
        break

    productos = r.json()

    if not productos:
        print("📭 Sin más productos")
        break

    print(f"📦 Página {pagina} -> {len(productos)} productos")

    for p in productos:
        try:
            nombre = p.get("name", "").lower()

            # 🔥 filtro carnicería
            if not any(x in nombre for x in ["carne", "pollo", "cerdo", "vacuno", "costilla", "filete"]):
                continue

            precio = p.get("prices", {}).get("price")
            if not precio:
                continue

            precio_final = int(precio) / 100

            datos.append({
                "producto": nombre,
                "precio": precio_final,
                "categoria": "carniceria",
                "supermercado": "Cugat",
                "fecha": time.strftime("%Y-%m-%d %H:%M:%S")
            })

            print(nombre, "->", precio_final)

        except:
            continue

    pagina += 1
    time.sleep(1)

print("📊 TOTAL CARNICERÍA:", len(datos))

# --- MONGO ---
try:
    client = MongoClient("mongodb", 27017)
    db = client["Canasta_db"]
    col = db["Retail_A"]

    if datos:
        col.insert_many(datos)
        print("💾 Guardado completo en MongoDB")
    else:
        print("⚠️ Sin datos")

except Exception as e:
    print("❌ Mongo error:", repr(e))

🌐 Scraping API COMPLETO Cugat
📦 Página 1 -> 100 productos
📦 Página 2 -> 100 productos
nuggets de pollo con figuras de dinosaurios super pollo, 360gr. -> 21.9
📦 Página 3 -> 100 productos
📦 Página 4 -> 100 productos
pollo asado elaboración propia cugat en bolsa unid. -> 96.9
nuggets la española pollo 2.5kg -> 109.9
📦 Página 5 -> 100 productos
costillitas de cerdo a la chilena, super cerdo 700gr. -> 86.9
churrasco vacuno receta del abuelo 120 gr -> 24.9
caldo en polvo sabor carne maggi, 35g -> 6.99
📦 Página 6 -> 100 productos
caldo en polvo sabor costilla maggi, 35g -> 6.99
📦 Página 7 -> 100 productos
pechuga entera pollo a granel super, kg -> 36.9
entraña de cerdo iqf super cerdo, 800gr. -> 106.9
cubitos de pulpa de cerdo congelados super cerdo, 500gr. -> 52.9
costillar de cerdo chileno super cerdo, 1kg -> 126.9
pechuga de pollo deshuesada congelada, kg -> 64.9
tiritas de pulpa cerdo iqf congelado super cerdo, 500gr. -> 52.9
pernil mano de cerdo a granel, cerdito vara, kg -> 29.9
pernil 